In [1]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
botname = 'ds6600'
version = '0.0'
purpose = 'Class demo'
email = 'wft5tv@virginia.edu'
useragent = f'{botname}/{version} ({purpose}; {email}) python-requests/{requests.__version__}'
headers = {'User-Agent': useragent}

In [ ]:
house_committees_url = "https://clerk.house.gov/committees/"

r = requests.get(house_committees_url, headers=headers)
r

In [5]:
r.text

'<!DOCTYPE html>\r\n<html lang="en" class="no-js">\r\n<head><script type="text/javascript">window.NREUM||(NREUM={});NREUM.info = {"beacon":"gov-bam.nr-data.net","errorBeacon":"gov-bam.nr-data.net","licenseKey":"9993044f26","applicationID":"1051873646","transactionName":"Z1xRYBQHCxYFAEQNCl4WfmIlSSYKCQ5ZEBFVXEAbLwgBABw=","queueTime":0,"applicationTime":268,"agent":"","atts":""}</script><script type="text/javascript">(window.NREUM||(NREUM={})).init={privacy:{cookies_enabled:true},ajax:{deny_list:["gov-bam.nr-data.net"]},distributed_tracing:{enabled:true}};(window.NREUM||(NREUM={})).loader_config={agentID:"1120208265",accountID:"561493",trustKey:"561493",xpid:"UQAGVV9QGwIAUVBaAAIBUlI=",licenseKey:"9993044f26",applicationID:"1051873646",browserID:"1120208265"};;/*! For license information please see nr-loader-spa-1.322.0.min.js.LICENSE.txt */\n(()=>{var e={8122(e,t,r){"use strict";r.d(t,{a:()=>i});var n=r(944);function i(e,t){try{if(!e||"object"!=typeof e)return(0,n.R)(3);if(!t||"object"!=t

In [6]:
## Beautiful soup doesn't get the data, it is strictly a local parser

In [8]:
mysoup = BeautifulSoup(r.text, 'html.parser')
#mysoup

Anatomy of HTML

```
<tag attribute="value"> navigable string </tag>

```

We're going to use "findall" so we can get as close to our data as possible.
- What do they have in common that other things don't have?

In [11]:
mylist = mysoup.find_all('a', attrs={'class':'library-committeePanel-subItems'})

In [ ]:
{'url': "https://clerk.house.gov" + mylist[0]['href'],
 'committee_name': mylist[0].text.strip()}

{'url': 'https://clerk.house.gov/committees/AG22',
 'committee_name': 'Commodity Markets, Digital Assets, and Rural Development'}

In [19]:
committee_df = pd.DataFrame([{'url': "https://clerk.house.gov" + x['href'],
 'committee_name': x.text.strip()} for x in mylist])

In [20]:
committee_df

,url,committee_name
0,https://clerk.house.gov/committees/AG22,"Commodity Markets, Digital Assets, and Rural D..."
1,https://clerk.house.gov/committees/AG14,"Conservation, Research, and Biotechnology"
2,https://clerk.house.gov/committees/AG15,Forestry and Horticulture
3,https://clerk.house.gov/committees/AG16,"General Farm Commodities, Risk Management, and..."
4,https://clerk.house.gov/committees/AG29,"Livestock, Dairy, and Poultry"
...,...,...
105,https://clerk.house.gov/committees/IG04,Defense Intelligence and Overhead Architecture
106,https://clerk.house.gov/committees/IG06,National Intelligence Enterprise
107,https://clerk.house.gov/committees/IG02,National Security Agency and Cyber
108,https://clerk.house.gov/committees/IG11,Open Source Intelligence


In [22]:
url = committee_df['url'][0]

In [23]:
r = requests.get(url, headers=headers)
r

<Response [200]>

In [24]:
mysoup = BeautifulSoup(r.text, 'html.parser')

In [28]:
mylist = mysoup.find_all('a', attrs = {'title': 'View Profile'})
## we want to get bioguide id, not member name, because it does entity resolution better

In [31]:
bioguideIds = [x['href'].replace('/Members/', '') for x in mylist]

In [32]:
## Now what do we do if we want the subcommittee name and the bioguide ID paired together?

In [ ]:
## this is part of the dataframe, next we need to loop over all of them.
pd.DataFrame({'committee': committee_df['committee_name'][0],
             'bioguideId': bioguideIds})

,committee,bioguideId
0,"Commodity Markets, Digital Assets, and Rural D...",J000301
1,"Commodity Markets, Digital Assets, and Rural D...",R000612
2,"Commodity Markets, Digital Assets, and Rural D...",L000491
3,"Commodity Markets, Digital Assets, and Rural D...",S001189
4,"Commodity Markets, Digital Assets, and Rural D...",R000603
5,"Commodity Markets, Digital Assets, and Rural D...",M000871
6,"Commodity Markets, Digital Assets, and Rural D...",C001039
7,"Commodity Markets, Digital Assets, and Rural D...",F000475
8,"Commodity Markets, Digital Assets, and Rural D...",N000193
9,"Commodity Markets, Digital Assets, and Rural D...",B001327


In [35]:
def congress_spider(url, committee):
    print(committee)
    r = requests.get(url, headers=headers)
    mysoup = BeautifulSoup(r.text, 'html.parser')
    mylist = mysoup.find_all('a', attrs = {'title': 'View Profile'})
    bioguideIds = [x['href'].replace('/Members/', '') for x in mylist]
    
    return {'committee': committee, 'bioguideId': bioguideIds}

In [36]:
full_committee_list = [congress_spider(u, c)
                        for u, c in zip(committee_df['url'], committee_df['committee_name'])]

Commodity Markets, Digital Assets, and Rural Development
Conservation, Research, and Biotechnology
Forestry and Horticulture
General Farm Commodities, Risk Management, and Credit
Livestock, Dairy, and Poultry
Nutrition and Foreign Agriculture
Agriculture, Rural Development, Food and Drug Administration, and Related Agencies
Commerce, Justice, Science, and Related Agencies
Defense
Energy and Water Development and Related Agencies
Financial Services and General Government
Homeland Security
Interior, Environment, and Related Agencies
Labor, Health and Human Services, Education, and Related Agencies
Legislative Branch
Military Construction, Veterans Affairs, and Related Agencies
National Security, Department of State, and Related Programs
Transportation, Housing and Urban Development, and Related Agencies
Cyber, Information Technologies, and  Innovation
Intelligence and Special Operations
Military Personnel
Readiness
Seapower and Projection Forces
Strategic Forces
Tactical Air and Land For

In [44]:
list_of_dfs = [pd.DataFrame(x) for x in full_committee_list]

In [45]:
full_df = pd.concat(list_of_dfs)

In [ ]:
full_df ## this is the entire set of congress people and their associated committees

,committee,bioguideId
0,"Commodity Markets, Digital Assets, and Rural D...",J000301
1,"Commodity Markets, Digital Assets, and Rural D...",R000612
2,"Commodity Markets, Digital Assets, and Rural D...",L000491
3,"Commodity Markets, Digital Assets, and Rural D...",S001189
4,"Commodity Markets, Digital Assets, and Rural D...",R000603
...,...,...
7,Open Source Intelligence,C001121
8,Open Source Intelligence,Q000023
9,Open Source Intelligence,C001068
0,Oversight and Investigations,J000304
